## 09 - Feature requests

What do users ask the app to add? I find the reviews that ask for something new with three methods and check them against my own labels.

In [1]:
# --- IMPORTS ---
import os
import re
import subprocess

import pandas as pd
from openpyxl.styles import Alignment
from openpyxl.worksheet.datavalidation import DataValidation

In [2]:
# --- SETTINGS ---
# Reviews drawn from each stratum (reviews that match the rules, and reviews that do not)
SAMPLE_PER_STRATUM = 200
# Fixed seed for the sample and its order, so the same sample comes out on every run
SAMPLE_SEED = 42
# The coding files. labeling/ is in .gitignore, because the files contain review texts.
SAMPLE_PATH = "../labeling/fr_sample.xlsx"
KEY_PATH = "../labeling/fr_key.csv"

In [3]:
# --- LOAD DATA ---
# The cleaned reviews, Italian only, as in 06. The rules read the original text, not text_clean:
# the cleaning of 03 removes apostrophes, which patterns such as "non c'è" and "l'opzione" need.
clean_df = pd.read_csv("../data/processed/trenord_reviews_clean.csv")
italian_df = clean_df[clean_df["language"] == "it"].copy().reset_index(drop=True)

# Reviews with no text cannot match a rule and cannot be coded, so they are counted
n_empty = 0
for text in italian_df["text"]:
    if not isinstance(text, str) or len(text.strip()) == 0:
        n_empty += 1

print(f"Italian reviews        : {len(italian_df):,}")
print(f"Reviews with empty text: {n_empty}")

Italian reviews        : 6,951
Reviews with empty text: 0


## Part 1 - Rules, sample and coding file

The rules below are fixed before any labelling and are not changed after looking at my labels. If I changed them after seeing which reviews they miss, they would fit this sample and their measured precision and recall would be too optimistic. A review matches if its text contains at least one of the patterns.

In [4]:
# --- THE RULES ---
# Regular expressions searched in the lowercase text. \b is a word boundary, so "vorrei" does
# not match inside a longer word. Alternatives that end with an apostrophe have no \b after it.
# Accented letters also accept the grave accent, or an apostrophe in place of the accent,
# because both spellings are common: "perchè", "possibilita'", "c'e'", "puo'".
REQUEST_PATTERNS = [
    r"\bvorrei\b",
    r"\bvorremmo\b",
    r"\bmi piacerebbe\b",
    r"\bci vorrebbe\b",
    r"\bservirebbe\b",
    r"\bsarebbe (utile|bello|comodo|carino|ottimo|meglio|opportuno|il caso|da)\b",
    r"\bsi potrebbe (aggiungere|mettere|inserire|avere|fare)\b",
    r"\bpotreste (aggiungere|mettere|inserire|fare|dare)\b",
    r"\bdovreste (aggiungere|mettere|inserire|fare|dare)\b",
    r"\baggiungete\b",
    r"\binserite\b",
    r"\baggiungere (la\b|il\b|una\b|un\b|l')",
    r"\bperch[eèé]'? non (mettete|aggiungete|fate|inserite|si pu[oò]'?)\b",
    r"\bmanca (la possibilit[aà]'?|l'opzione|la funzione|una funzione|un'opzione)\b",
    r"\bnon c'[eèé]'? (la possibilit[aà]'?|l'opzione|la funzione|modo di)\b",
    r"\bpossibilit[aà]'? di\b",
    r"\bopzione per\b",
    r"\bsuggerisco\b",
    r"\bpropongo\b",
    r"\bconsiglio di (aggiungere|inserire|mettere)\b",
]

print(f"Patterns: {len(REQUEST_PATTERNS)}")

Patterns: 20


In [5]:
# --- PREPARE THE TEXTS FOR THE RULES ---
# A lowercase copy of each text, with the typographic apostrophes replaced by the plain one,
# so the rules see the same apostrophe whatever keyboard was used. The original text is not changed.
rule_texts = []
for text in italian_df["text"]:
    if not isinstance(text, str):
        text = ""
    text = text.lower()
    text = text.replace("’", "'")
    text = text.replace("‘", "'")
    rule_texts.append(text)

print(f"Texts prepared: {len(rule_texts):,}")

Texts prepared: 6,951


In [6]:
# --- APPLY THE RULES ---
# A review matches as soon as one pattern is found in it
rule_matches = []
for text in rule_texts:
    is_match = False
    for pattern in REQUEST_PATTERNS:
        if re.search(pattern, text):
            is_match = True
            break
    rule_matches.append(is_match)

italian_df["rule_match"] = rule_matches

n_match = italian_df["rule_match"].sum()
print(f"Reviews that match: {n_match:,} of {len(italian_df):,} ({n_match / len(italian_df) * 100:.1f}%)")

Reviews that match: 300 of 6,951 (4.3%)


In [7]:
# --- REVIEWS PER PATTERN ---
# How many reviews each pattern finds. A review can contain more than one pattern, so the
# counts add up to more than the reviews that match.
rows = []
for pattern in REQUEST_PATTERNS:
    n_reviews = 0
    for text in rule_texts:
        if re.search(pattern, text):
            n_reviews += 1
    rows.append({"pattern": pattern, "reviews": n_reviews})

pattern_counts = pd.DataFrame(rows).set_index("pattern")
pattern_counts

,reviews
pattern,
\bvorrei\b,35
\bvorremmo\b,0
\bmi piacerebbe\b,10
\bci vorrebbe\b,2
\bservirebbe\b,2
\bsarebbe (utile|bello|comodo|carino|ottimo|meglio|opportuno|il caso|da)\b,62
\bsi potrebbe (aggiungere|mettere|inserire|avere|fare)\b,1
\bpotreste (aggiungere|mettere|inserire|fare|dare)\b,2
\bdovreste (aggiungere|mettere|inserire|fare|dare)\b,4


In [8]:
# --- THE STRATA AND THEIR SIZES ---
# The sample is drawn separately from the reviews that match and from those that do not.
# These two sizes are the weights for the estimates on all reviews in the next parts.
n_no_match = len(italian_df) - n_match
stratum_table = pd.DataFrame({
    "stratum": ["match", "no match"],
    "reviews": [n_match, n_no_match],
    "share (%)": [round(n_match / len(italian_df) * 100, 1), round(n_no_match / len(italian_df) * 100, 1)],
}).set_index("stratum")
stratum_table

,reviews,share (%)
stratum,,
match,300,4.3
no match,6651,95.7


### The sample

I code 200 reviews that match and 200 that do not, instead of 400 drawn from all reviews. A plain random sample would contain few matching reviews, too few to measure how often the rules are right, while the non-matching half shows the requests the rules miss. Because the halves do not have the real proportions, estimates for all reviews weight each half by the size of its stratum.

In [9]:
# --- DRAW THE SAMPLE ---
# The same number of reviews from each stratum, drawn at random with a fixed seed.
# If fewer reviews match than the sample size, all of them are taken.
matching_df = italian_df[italian_df["rule_match"]]
non_matching_df = italian_df[~italian_df["rule_match"]]

n_from_matching = min(SAMPLE_PER_STRATUM, len(matching_df))
if n_from_matching < SAMPLE_PER_STRATUM:
    print(f"Only {len(matching_df)} reviews match: all of them are in the sample")

matching_sample = matching_df.sample(n=n_from_matching, random_state=SAMPLE_SEED)
non_matching_sample = non_matching_df.sample(n=SAMPLE_PER_STRATUM, random_state=SAMPLE_SEED)
sample_df = pd.concat([matching_sample, non_matching_sample])

print(f"Sampled from the reviews that match       : {len(matching_sample)}")
print(f"Sampled from the reviews that do not match: {len(non_matching_sample)}")
print(f"Reviews in the sample                     : {len(sample_df)}")
print(f"Distinct reviews                          : {sample_df['review_id'].nunique()}")

Sampled from the reviews that match       : 200
Sampled from the reviews that do not match: 200
Reviews in the sample                     : 400
Distinct reviews                          : 400


In [10]:
# --- SHUFFLE THE ORDER ---
# Without shuffling, the matching reviews would come first and the coder would know the rule
# result from the position. sample(frac=1) returns all rows in random order.
sample_df = sample_df.sample(frac=1, random_state=SAMPLE_SEED).reset_index(drop=True)

# The id is given after shuffling, so it says nothing about the rule result
sample_df["sample_id"] = range(1, len(sample_df) + 1)

# Longest run of reviews with the same rule result in a row, to check they are not in blocks
longest_run = 1
current_run = 1
sample_matches = sample_df["rule_match"].tolist()
for i in range(1, len(sample_matches)):
    if sample_matches[i] == sample_matches[i - 1]:
        current_run += 1
    else:
        current_run = 1
    if current_run > longest_run:
        longest_run = current_run

print(f"Longest run of the same rule result in a row: {longest_run}")

Longest run of the same rule result in a row: 16


In [11]:
# --- CODEBOOK ---
# What each answer means. It goes to its own sheet of the coding file.
CODEBOOK = {
    "yes": "the review asks for a feature or a change that the app does not offer today "
           "(for example invoices, purchase history, another payment method, refunds in the app, "
           "notifications for a line).",
    "no": "everything else, including reports that something does not work, generic requests to "
          "improve the app, missing stations or wrong data, praise, complaints about trains.",
    "mixed": "if a review reports a problem and also asks for something new, answer yes.",
    "what_is_requested": "a few words on what is asked, in Italian or English "
                         "(for example \"fattura\", \"storico acquisti\"). Leave empty when the answer is no.",
}

print(f"Codebook rows: {len(CODEBOOK)}")

Codebook rows: 4


In [12]:
# --- WRITE THE CODING FILES (only the first time) ---
# If the Excel file is already there, nothing is written, so a rerun never overwrites the coding.
if os.path.exists(SAMPLE_PATH):
    print("labeling/fr_sample.xlsx already exists: no file was written, the coding is kept")
else:
    os.makedirs("../labeling", exist_ok=True)

    # Sheet "coding": the review and two empty columns for my answer. The rule result is not here.
    coding_sheet = sample_df[["sample_id", "text"]].copy()
    coding_sheet["is_request"] = None
    coding_sheet["what_is_requested"] = None

    # Sheet "codebook": one row per entry with its definition
    codebook_rows = []
    for entry, definition in CODEBOOK.items():
        codebook_rows.append({"entry": entry, "definition": definition})
    codebook_sheet = pd.DataFrame(codebook_rows)

    with pd.ExcelWriter(SAMPLE_PATH, engine="openpyxl") as writer:
        coding_sheet.to_excel(writer, sheet_name="coding", index=False)
        codebook_sheet.to_excel(writer, sheet_name="codebook", index=False)

        # Dropdown with the two answers in the is_request column
        worksheet = writer.sheets["coding"]
        dropdown = DataValidation(type="list", formula1='"yes,no"', allow_blank=True, showErrorMessage=True)
        worksheet.add_data_validation(dropdown)
        dropdown.add(f"C2:C{len(coding_sheet) + 1}")

        # Wide, wrapped text column, so long reviews can be read without resizing
        worksheet.column_dimensions["B"].width = 100
        worksheet.column_dimensions["C"].width = 12
        worksheet.column_dimensions["D"].width = 40
        for row in worksheet.iter_rows(min_row=2, min_col=2, max_col=2):
            for cell in row:
                cell.alignment = Alignment(wrap_text=True, vertical="top")

        codebook_worksheet = writer.sheets["codebook"]
        codebook_worksheet.column_dimensions["A"].width = 20
        codebook_worksheet.column_dimensions["B"].width = 100
        for row in codebook_worksheet.iter_rows(min_row=2, min_col=2, max_col=2):
            for cell in row:
                cell.alignment = Alignment(wrap_text=True, vertical="top")

    # The hidden key: the rule result of each sample_id. It stays closed until the coding is done.
    key_df = sample_df[["sample_id", "review_id", "rule_match"]]
    key_df.to_csv(KEY_PATH, index=False, encoding="utf-8-sig")

    print("Created labeling/fr_sample.xlsx and labeling/fr_key.csv")

Created labeling/fr_sample.xlsx and labeling/fr_key.csv


In [13]:
# --- CHECK THE CODING FILES ---
# The files are read back from disk. Only counts and True or False are printed, so the
# texts and the key stay hidden.
coding_back = pd.read_excel(SAMPLE_PATH, sheet_name="coding")
codebook_back = pd.read_excel(SAMPLE_PATH, sheet_name="codebook")
key_back = pd.read_csv(KEY_PATH)

# The coding sheet must have exactly these columns: no rule result in it
expected_columns = ["sample_id", "text", "is_request", "what_is_requested"]
right_columns = coding_back.columns.tolist() == expected_columns

# Same reviews in the same order as the sample drawn above. If the data has changed since
# the files were made, this is False and the files describe an older sample.
texts_in_file = coding_back["text"].fillna("").tolist()
texts_in_sample = sample_df["text"].fillna("").tolist()
same_texts = texts_in_file == texts_in_sample

# Same ids in the coding sheet and in the key, and the same reviews in the key as in the sample
same_ids = coding_back["sample_id"].tolist() == key_back["sample_id"].tolist()
same_reviews = key_back["review_id"].tolist() == sample_df["review_id"].tolist()

n_coded = coding_back["is_request"].notna().sum()

print(f"Rows in the coding sheet        : {len(coding_back)}")
print(f"Rows in the codebook sheet      : {len(codebook_back)}")
print(f"Rows in the key                 : {len(key_back)}")
print(f"Coding sheet without rule result: {right_columns}")
print(f"Same texts as the sample        : {same_texts}")
print(f"Same ids in sheet and key       : {same_ids}")
print(f"Same reviews in key and sample  : {same_reviews}")
print(f"Reviews coded so far            : {n_coded}")

Rows in the coding sheet        : 400
Rows in the codebook sheet      : 4
Rows in the key                 : 400
Coding sheet without rule result: True
Same texts as the sample        : True
Same ids in sheet and key       : True
Same reviews in key and sample  : True
Reviews coded so far            : 0


In [14]:
# --- CHECK THAT THE CODING FILES STAY OUT OF GIT ---
# git check-ignore exits with 0 when a path is ignored. git diff --cached lists the staged
# files, which must not include anything in labeling/.
sample_ignored = subprocess.run(["git", "check-ignore", "-q", SAMPLE_PATH]).returncode == 0
key_ignored = subprocess.run(["git", "check-ignore", "-q", KEY_PATH]).returncode == 0

staged = subprocess.run(["git", "diff", "--cached", "--name-only", "--", "../labeling"],
                        capture_output=True, text=True).stdout
nothing_staged = staged.strip() == ""

print(f"fr_sample.xlsx ignored by git : {sample_ignored}")
print(f"fr_key.csv ignored by git     : {key_ignored}")
print(f"Nothing in labeling/ is staged: {nothing_staged}")

assert sample_ignored and key_ignored and nothing_staged, "A coding file could end up in git"

fr_sample.xlsx ignored by git : True
fr_key.csv ignored by git     : True
Nothing in labeling/ is staged: True
